In [4]:
!pip install transformers

In [5]:
from transformers import pipeline

sentiment_model = pipeline("sentiment-analysis",model = "distilbert-base-uncased-finetuned-sst-2-english")

D:\Laptop Backup\Softpro\Data Analytics\myenv\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

D:\Laptop Backup\Softpro\Data Analytics\myenv\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [13]:
text = ["I love Transformer","I love creating model, training them and evaluating that as well."]
neg_text = ["I find difficult working with dataset that causes my GPU to get out of cuda memory."]

out = sentiment_model(text)
neg_out = sentiment_model(neg_text)

for res in out:
    print(res)

print(neg_out)

{'label': 'POSITIVE', 'score': 0.9998635053634644}
{'label': 'POSITIVE', 'score': 0.9996770620346069}
[{'label': 'NEGATIVE', 'score': 0.9996919631958008}]


In [2]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [21]:
import pandas as pd
import torch
ds["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [6]:
df = pd.DataFrame(ds["train"])

In [7]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

D:\Laptop Backup\Softpro\Data Analytics\myenv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abhay\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

D:\Laptop Backup\Softpro\Data Analytics\myenv\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [18]:
model = model.to("cuda")

In [16]:
tokens = tokenizer(df.iloc[0]["text"][:500],return_tensors="pt").input_ids
out = model(tokens).logits[0,-1]

val = out.argmax().item()
model.config.id2label[val]

'NEGATIVE'

In [31]:
def prediction(raw_text):
    tokens = tokenizer(raw_text[:512],return_tensors="pt").input_ids.to("cuda")
    with torch.no_grad():
        out = model(tokens).logits 
    predicted_class_id = out.argmax().item()
    final_class = model.config.id2label[predicted_class_id]

    return final_class

In [32]:
df["model_pred"] = df["text"].apply(prediction)

In [34]:
df

,text,label,model_pred
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,NEGATIVE
1,"""I Am Curious: Yellow"" is a risible and preten...",0,NEGATIVE
2,If only to avoid making this type of film in t...,0,NEGATIVE
3,This film was probably inspired by Godard's Ma...,0,NEGATIVE
4,"Oh, brother...after hearing about this ridicul...",0,NEGATIVE
...,...,...,...
24995,A hit at the time but now better categorised a...,1,NEGATIVE
24996,I love this movie like no other. Another time ...,1,POSITIVE
24997,This film and it's sequel Barry Mckenzie holds...,1,POSITIVE
24998,'The Adventures Of Barry McKenzie' started lif...,1,NEGATIVE
